In [12]:
!pip install numpy torch einops scikit-learn matplotlib cmocean transformers

  Using cached transformers-5.10.2-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.18.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.5.9-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.26.7-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached click-8.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.5.0-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached markdown

In [1]:
import os
import torch

import sys
sys.path.append(os.path.abspath('../'))

from FourCastNet.networks.afnonet_wf import AFNONet_Seq2Seq
from common_utils.config import Config

from common_utils.dataloader import create_datasets_lite, create_dataloaders

# 0. config
args_config = '../data_lanl_fire/config.yaml'
config = Config(filepath=args_config, override_from_env=False, override_from_args=False)
config.in_channels = 4

# # 1. Instantiate the model structure
model = AFNONet_Seq2Seq(
    img_size=(config.image_size, config.image_size), patch_size=(config.patch_size, config.patch_size),
    in_chans=config.in_channels, out_chans=config.out_channels,
    context_len=config.context_len, temporal_patch_size=config.temporal_patch_size,
    embed_dim=config.embed_dim, depth=config.depth, num_blocks=config.num_blocks
)


# 2. Load your weights
model_path = '../data_lanl_fire/best_model.pth'
weights = torch.load(model_path, map_location=torch.device('cpu'))
model.load_state_dict(weights['state_dict'])

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<All keys matched successfully>

In [3]:
project_dir = '/home/jovyan/work/pgml-scil'
ddp_rank = 1

print('Creating datasets and dataloaders...')
train_dataset, val_dataset, test_dataset = create_datasets_lite(config, project_dir, ddp_rank, config.ddp_world_size)

print('Creating dataloaders...')
train_loader, val_loader, test_loader = create_dataloaders(
    train_dataset, val_dataset, test_dataset, config, test_batch_size=1
)

# print('Running inference on test data...')
# # input_test = val_loader.dataset.inputs[0:1,0:6,:,:,:]
# input_test = test_loader.dataset[0][0][0:6,:,:,:].unsqueeze(0)

Creating datasets and dataloaders...


FileNotFoundError: [Errno 2] No such file or directory: '/home/pgmlvol/data/data_lite/'

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots(1,4, figsize=(20,5))
_ = ax[0].hist(test_loader.dataset[0][0][:,0,:,:].flatten(), bins=100, log=True, label='fuel density')
ax[0].set_title('fuel density')
_ = ax[1].hist(test_loader.dataset[0][0][:,1,:,:].flatten(), bins=100, log=True, label='wind component 1')
ax[1].set_title('wind component 1')
_ = ax[2].hist(test_loader.dataset[0][0][:,2,:,:].flatten(), bins=100, log=True, label='wind component 2')
ax[2].set_title('wind component 2')
_ = ax[3].hist(test_loader.dataset[0][0][:,3,:,:].flatten(), bins=100, log=True, label='source map')
ax[3].set_title('source map')

In [4]:
# Generate gif for the predictions
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io

from tqdm import tqdm

def images_to_gif(images, output_path, duration=200, cmap='viridis', vmin=None, vmax=None, start_index=0):
    """
    Convert a list of 2D numpy arrays to an animated GIF.
    
    Args:
        images: list of 2D numpy arrays
        output_path: path to save the GIF (e.g., 'output.gif')
        duration: milliseconds per frame
        cmap: matplotlib colormap
        vmin/vmax: colormap value range (uses global min/max if None)
    """
    frames = []
    vmin = vmin if vmin is not None else min(img.min() for img in images)
    vmax = vmax if vmax is not None else max(img.max() for img in images)

    for i, img in tqdm(enumerate(images), total=len(images), desc="Creating GIF frames"):
        fig, ax = plt.subplots()
        ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(f'Time step {i + start_index}')
        ax.axis('off')

        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
        plt.close(fig)
        buf.seek(0)
        frames.append(Image.open(buf).convert('RGBA'))

    frames[0].save(
        output_path,
        save_all=True,
        append_images=frames[1:],
        duration=duration,
        loop=0  # 0 = loop forever
    )
    print(f"Saved GIF to {output_path}")

In [5]:
# Calculate predictions for each time step
model.eval()
with torch.no_grad():
    predicted_images = []
    for t in tqdm(range(6,50), desc="Calculating predictions"):
        input_t = test_loader.dataset[0][0][t-6:t,:,:,:].unsqueeze(0)
        pred_t = model(input_t)[0,0,0,:,:].detach().numpy()
        predicted_images.append(pred_t)

Calculating predictions:   0%|          | 0/44 [00:00<?, ?it/s]


NameError: name 'test_loader' is not defined

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

black_white_green = LinearSegmentedColormap.from_list(
    'black_white_green',
    ['black', 'white', 'green']
)

# Save the predictions as a GIF
images_to_gif(predicted_images, 'predictions.gif', duration=200, vmin=0, vmax=0.7, cmap=black_white_green, start_index=6)

In [ ]:
# Plot each input channel for each timestep and create a gif
test_loader.dataset[0][0].shape

fuels = [test_loader.dataset[0][0][t,0,:,:].numpy() for t in range(50)]
windcos = [test_loader.dataset[0][0][t,1,:,:].numpy() for t in range(50)]
windsin = [test_loader.dataset[0][0][t,2,:,:].numpy() for t in range(50)]
sources = [test_loader.dataset[0][0][t,3,:,:].numpy() for t in range(50)]

images_to_gif(fuels, 'fuels.gif', duration=200, vmin=0, vmax=0.7, cmap=black_white_green, start_index=0)
images_to_gif(windcos, 'windcos.gif', duration=200, vmin=-1, vmax=1, cmap='bwr', start_index=0)
images_to_gif(windsin, 'windsin.gif', duration=200, vmin=-1, vmax=1, cmap='bwr', start_index=0)
images_to_gif(sources, 'sources.gif', duration=200, vmin=0, vmax=0.7, cmap=black_white_green, start_index=0)


In [ ]:
# Generate time-lapse for the firetec dataset.

fig, ax = plt.subplots(1,2, figsize=(10,5))

folder = '/home/pgmlvol/tcaglar/pgml-scil/data/firetec-data-test/'

firetec_fuels = np.load(os.path.join(folder, 'output.12000.npy'))
# firetec_fuels.resize(296,296)

img = Image.fromarray(firetec_fuels)
# ax[0].imshow(np.array(img.resize((296,296))), cmap=black_white_green, vmin=0, vmax=1)
# ax[0].set_title('Resized fuels')
ax[0].imshow(firetec_fuels[50:(50+296), 100:(100+296)], cmap=black_white_green, vmin=0, vmax=1)
ax[0].set_title('Cropped fuels')


ax[1].imshow(firetec_fuels, cmap=black_white_green, vmin=0, vmax=1)
ax[1].set_title('Original fuels')

ax[0].plot([140,160], [170,170], 'k-')
ax[0].plot([140,160], [169,169], 'k-')


In [ ]:
firetec_fuels_cropped = []
for t in tqdm(range(1000,64000,1000), desc="Processing firetec fuels"):
    firetec_fuels = np.load(os.path.join(folder, f'output.{t}.npy'))
    firetec_fuels = firetec_fuels[50:(50+296), 100:(100+296)]
    firetec_fuels_cropped.append(firetec_fuels)

# Save the predictions as a GIF
images_to_gif(firetec_fuels_cropped, 'firetec_fuels_cropped.gif', duration=200, vmin=0, vmax=1, cmap=black_white_green, start_index=0)

In [ ]:
# Generate source map for the firetec dataset

fig, ax = plt.subplots(1,4, figsize=(20,5))

folder = '/home/pgmlvol/tcaglar/pgml-scil/data/firetec-data-test/'

firetec_fuels = np.load(os.path.join(folder, 'output.12000.npy'))
# firetec_fuels.resize(296,296)

img = Image.fromarray(firetec_fuels)
# ax[0].imshow(np.array(img.resize((296,296))), cmap=black_white_green, vmin=0, vmax=1)
# ax[0].set_title('Resized fuels')
ax[0].imshow(firetec_fuels[50:(50+296), 100:(100+296)], cmap=black_white_green, vmin=0, vmax=1)
ax[0].set_title('Cropped fuels')

windcos = np.zeros((296,296))
ax[1].imshow(windcos, cmap='bwr', vmin=-1, vmax=1)
ax[1].set_title('Wind x-component')

windsin = np.ones((296,296))
ax[2].imshow(windsin, cmap='bwr', vmin=-1, vmax=1)
ax[2].set_title('Wind y-component')

source_map = np.ones((296,296))
# source_map = firetec_fuels[50:(50+296), 100:(100+296)].copy()
source_map[169, 140:160] = 0
ax[3].imshow(source_map, cmap=black_white_green, vmin=0, vmax=1)
ax[3].set_title('Source map')



In [ ]:
firetec_inputs = np.zeros((64,4,296,296))
for i, time in tqdm(enumerate(range(1000,64000,1000)), desc="Processing firetec inputs", total=64):
    firetec_fuels = np.load(os.path.join(folder, f'output.{time}.npy'))
    firetec_fuels = firetec_fuels[50:(50+296), 100:(100+296)]
    firetec_inputs[i,0,:,:] = firetec_fuels
    firetec_inputs[i,1,:,:] = windcos
    firetec_inputs[i,2,:,:] = windsin
    firetec_inputs[i,3,:,:] = source_map

In [ ]:
firetec_predictions = []
model.eval()
with torch.no_grad():
    firetec_inputs_tensor = torch.from_numpy(firetec_inputs).float()
    for t in tqdm(range(6,64), desc="Calculating firetec predictions"):
        input_t = firetec_inputs_tensor[t-6:t,:,:,:].unsqueeze(0)
        pred_t = model(input_t)
        firetec_predictions.append(pred_t[0][0][0].numpy())


In [ ]:
# Save the predictions as a GIF
images_to_gif(firetec_predictions, 'firetec_predictions.gif', duration=200, vmin=0, vmax=1, cmap=black_white_green, start_index=6)

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib as mpl
import io

def combine_gifs_side_by_side(gifpath1, gifpath2, outputpath, duration=200, loop=0, gap=10,
                               cmap=None, vmin=0, vmax=1,
                               title1='original', title2='predicted', title_height=30,
                               skip1=0, skip2=0):
    gif1 = Image.open(gifpath1)
    gif2 = Image.open(gifpath2)

    # Skip initial frames
    for _ in range(skip1):
        gif1.seek(gif1.tell() + 1)
    for _ in range(skip2):
        gif2.seek(gif2.tell() + 1)

    # Pre-render colorbar once (same for all frames)
    cb_img = None
    if cmap is not None:
        colorbar_width = 60
        fig, ax = plt.subplots(figsize=(0.6, 3))
        norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
        mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='vertical')
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0.05, dpi=100)
        plt.close(fig)
        buf.seek(0)
        cb_img = Image.open(buf).convert('RGBA')

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", size=16)
    except Exception:
        font = ImageFont.load_default()

    frames = []
    try:
        while True:
            f1 = gif1.copy().convert('RGBA')
            f2 = gif2.copy().convert('RGBA')

            total_width = f1.width + gap + f2.width
            total_height = title_height + f1.height
            if cb_img is not None:
                cb_resized = cb_img.resize((colorbar_width, f1.height), Image.LANCZOS)
                total_width += gap + colorbar_width

            combined = Image.new('RGBA', (total_width, total_height), (255, 255, 255, 255))
            draw = ImageDraw.Draw(combined)

            # Draw titles
            for text, x_center in [(title1, f1.width // 2), (title2, f1.width + gap + f2.width // 2)]:
                bbox = draw.textbbox((0, 0), text, font=font)
                text_w = bbox[2] - bbox[0]
                draw.text((x_center - text_w // 2, (title_height - (bbox[3] - bbox[1])) // 2),
                          text, fill=(0, 0, 0, 255), font=font)

            # Paste frames below titles
            combined.paste(f1, (0, title_height))
            combined.paste(f2, (f1.width + gap, title_height))

            if cb_img is not None:
                combined.paste(cb_resized, (f1.width + gap + f2.width + gap, title_height))

            frames.append(combined)

            gif1.seek(gif1.tell() + 1)
            gif2.seek(gif2.tell() + 1)
    except EOFError:
        pass

    frames[0].save(
        outputpath,
        save_all=True,
        append_images=frames[1:],
        duration=duration,
        loop=loop
    )
    print(f"Saved combined GIF to {outputpath}")


In [ ]:
combine_gifs_side_by_side(
    'firetec_fuels_cropped.gif', 'firetec_predictions.gif', 'firetec_comparison.gif',
    duration=200, loop=0, gap=10, cmap=black_white_green, vmin=0, vmax=1,
    title1='original', title2='predicted', skip1=6
)


In [ ]:
combine_gifs_side_by_side(
    'fuels.gif', 'predictions.gif', 'pgml_test_comparison.gif',
    duration=200, loop=0, gap=10, cmap=black_white_green, vmin=0, vmax=1,
    title1='original', title2='predicted', skip1=6
)